In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision 

#model is used on a series of input images and outputs and results in 10 labels (0-9) defined by features
#  each corresponding to a particular output. Used to classift new input images 

class Net(nn.Module):
    def __init__(self):
      super(Net, self).__init__()
      self.conv1 = nn.Conv2d(1, 6, 5)
      self.conv2 = nn.Conv2d(6, 16, 5)
      # an affine operation: y = Wx + b
      self.fc1 = nn.Linear(16 * 5 * 5, 120)  # 5*5 from image dimension
      self.fc2 = nn.Linear(120, 84)
      self.fc3 = nn.Linear(84, 10)


    def forward(self, input):
        #layer one, conv.
        c1 = F.relu(self.conv1(input))
        #layer two, subsample
        s2 = F.max_pool2d(c1, (2, 2))
        #layer three, conv
        c3 = F.relu(self.conv2(s2))
        #layer four, subsample
        s4 = F.max_pool2d(c3, 2)
        #layer five, flattened
        s4 = torch.flatten(s4, 1)
        #fully connected layer F5: (N, 16*5*5) Tensor input, and outputs a (N, 120) Tensor,
        f5 = F.relu(self.fc1(s4))
        #fully connected layer F6: (N, 120) Tensor input,
        f6 = F.relu(self.fc2(f5))
        #OUTPUT: (N, 84) Tensor input, and FINAL OUTPUT: a (N, 10) Tensor
        output = self.fc3(f6)
        return output

net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [ ]:
params = list(net.parameters())
print(len(params)) #learnable parameters in network (0-9)
print(params[0].size())  # conv1's .weight

10
torch.Size([6, 1, 5, 5])


In [ ]:
#random input
input = torch.randn(1, 1, 32, 32)
out = net(input)
print(out)

tensor([[ 0.0011,  0.0387,  0.0114, -0.1140,  0.0601,  0.0047,  0.0775,  0.0807,
          0.0240,  0.0724]], grad_fn=<AddmmBackward0>)


In [5]:
#clears all current gradients
net.zero_grad()
#example of backpropagation with random gradients 
out.backward(torch.randn(1, 10))

In [ ]:
#LOSS FUNCTION: takes output and target then finds how far off the output is from the target.
#common example, mean sqaured error loss: MSE = 1/n * sum((output-target)^2)
output = net(input)
target = torch.randn(10)  # a dummy target, for example
target = target.view(1, -1)  # make it the same shape as output
criterion = nn.MSELoss() #defining MSE loss function
loss = criterion(output, target) #(output-target)^2
print(loss)

tensor(0.6568, grad_fn=<MseLossBackward0>)


In [ ]:
#.next_functions gives us the functions that created the loss and its ancestors
print(loss.grad_fn)  # MSELoss
print(loss.grad_fn.next_functions[0][0])  # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])  # ReLU

In [8]:
#backpropagting the loss: adjusting the weights to minimize the loss 
net.zero_grad()
print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)
loss.backward()
print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)


conv1.bias.grad before backward
None
conv1.bias.grad after backward
tensor([ 0.0129, -0.0014,  0.0161,  0.0141,  0.0057,  0.0092])


In [9]:
#update the weights of the network according to the formula w = w - learning_rate * gradient
learning_rate = 0.01
for f in net.parameters():
    with torch.no_grad():
        f -= f.grad * learning_rate

In [ ]:
import torch.optim as optim

#creating an automatic optimizer for updating the weights of the network. 
# The SGD optimizer implements stochastic gradient descent, the method used above
optimizer = optim.SGD(net.parameters(), lr=0.01)

#in training loop:
optimizer.zero_grad()   # zero the gradient buffers
output = net(input)
loss = criterion(output, target) #calculate loss
loss.backward() #backpropagate the loss
optimizer.step() #updates the weights according to the optimizer's update rule